In [17]:
%gui tk

In [30]:
import tkinter as tk
from tkinter import ttk, messagebox
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer


# train model from lab 7 -----------------------------------------

df = pd.read_csv("carprices4.csv")

# dummy variables model ------------------------------------------

dummies = pd.get_dummies(df['Car Model'], dtype=int)
merged = pd.concat([df, dummies], axis='columns')
final = merged.drop(['Car Model', 'Mercedez Benz C class'], axis='columns')

X = final.drop(['Sell Price($)'], axis='columns')
y = final['Sell Price($)']

model = LinearRegression()
model.fit(X, y)

feature_columns = list(X.columns)   # save order of features


# one-hot encoding model -----------------------------------------

X2 = df[['Car Model','Mileage','Age(yrs)']]
y2 = df['Sell Price($)']

ct = ColumnTransformer(
    transformers=[('encoder', OneHotEncoder(sparse_output=False), ['Car Model'])],
    remainder='passthrough'
)

X2 = ct.fit_transform(X2)
X2 = X2[:,1:] 

model2 = LinearRegression()
model2.fit(X2, y2)

# save learned categories from encoder (real car names)
ohe_categories = ct.named_transformers_['encoder'].categories_[0]


# GUI --------------------------------------------------------------

root = tk.Tk()
root.title("Car Price Predictor")
root.geometry("450x350")

# car model dropdown
ttk.Label(root, text="Car Model:").grid(row=0, column=0, sticky=tk.W, pady=5, padx=5)
car_model = tk.StringVar()
car_model_box = ttk.Combobox(root, textvariable=car_model,
                             values=df['Car Model'].unique().tolist(),
                             state="readonly")
car_model_box.grid(row=0, column=1, pady=5)

# mileage input
ttk.Label(root, text="Mileage (km):").grid(row=1, column=0, sticky=tk.W, pady=5, padx=5)
mileage_var = tk.StringVar()
ttk.Entry(root, textvariable=mileage_var).grid(row=1, column=1, pady=5)

# age input
ttk.Label(root, text="Age (years):").grid(row=2, column=0, sticky=tk.W, pady=5, padx=5)
age_var = tk.StringVar()
ttk.Entry(root, textvariable=age_var).grid(row=2, column=1, pady=5)

# encoding method radio buttons
encoding_method = tk.IntVar(value=0)  # 0 = Dummy, 1 = One-Hot
ttk.Label(root, text="Encoding Method:").grid(row=3, column=0, sticky=tk.W, pady=5, padx=5)
ttk.Radiobutton(root, text="Dummy Variables", variable=encoding_method, value=0).grid(row=3, column=1, sticky=tk.W)
ttk.Radiobutton(root, text="One-Hot Encoding", variable=encoding_method, value=1).grid(row=4, column=1, sticky=tk.W)

# result label
result_var = tk.StringVar(value="Enter details and click Predict")
ttk.Label(root, textvariable=result_var, font=("Arial", 12, "bold")).grid(row=6, column=0, columnspan=2, pady=20)

# prediction function
def predict_price():
    try:
        mileage = float(mileage_var.get())
        age = int(age_var.get())
        car = car_model.get()
        method = encoding_method.get()

        if method == 0:  # dummy Variables
            features = [0] * len(feature_columns)
            if "Mileage" in feature_columns:
                features[feature_columns.index("Mileage")] = mileage
            if "Age(yrs)" in feature_columns:
                features[feature_columns.index("Age(yrs)")] = age
            for col in feature_columns:
                if col == car:
                    features[feature_columns.index(col)] = 1
            price = model.predict([features])[0]

        else:  # one-hot Encoding
            # encode selected car against learned categories
            car_onehot = []

            for cat in ohe_categories[1:]:
                if car == cat:
                    car_onehot.append(1)
                else:
                    car_onehot.append(0)  

            features = car_onehot + [mileage, age]
            price = model2.predict([features])[0]

        result_var.set(f"Predicted Price: ${price:,.2f}")

    except ValueError:
        messagebox.showerror("Input Error", "Please enter valid numbers for Mileage and Age.")
    except Exception as e:
        messagebox.showerror("Error", str(e))

# buttons
ttk.Button(root, text="Predict", command=predict_price).grid(row=5, column=0, pady=10)
ttk.Button(root, text="Reset",
           command=lambda: [car_model.set(""), mileage_var.set(""), age_var.set(""),
                            result_var.set("Enter details and click Predict")]).grid(row=5, column=1, pady=10)

# in jupyter use %gui tk and do NOT call root.mainloop()


/Users/dani/Desktop/csc180/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
